# 15 - Sensitivity of the result to the quality filter / peak-detection settings (original windows)
Reviewer 2 comment 3. For 20 settings of the quality gate and of the peak detector (and of an optional band-pass filter) the subject-wise 5-fold cross-validation of the neural network is repeated and the share of AF and non-AF windows that is retained is reported next to the accuracy.

**Letter values (accuracy %, windows retained):** published settings (gate 0.8, prominence 15%, min. distance 0.4 s): 96.60, 1,920 windows (93.76% of AF, 89.24% of non-AF). Plausibility gate 100%: 98.23, 1,683 windows (79.07% AF, 81.92% non-AF); gate 60%: 92.99, 2,017 windows. Prominence 5%: 93.62, 2,047 windows; 35%: 97.78, 1,697 windows. Band-pass 0.5-8 Hz zero-phase: 93.81, 2,013 windows; causal: 97.50, 1,847 windows. Over the 20 settings the accuracy ranges from 89.02% to 98.23%.
The original list of the 20 settings is not available to me, so the grid below is a reconstruction around the values above; the entries that the letter quotes (marked `*`) must reproduce them. Please keep the grid unchanged and report differences.

Runtime about 20-40 min (100 trainings).

**How to run (Google Colab, the same place the other notebooks ran).** Folder `MyDrive/ppg_afib_project/` must contain the two PERform zips (`mimic_perform_af_csv.zip`, `mimic_perform_non_af_csv.zip`, searched recursively). Run the cells top to bottom with *Runtime -> Run all*. **Do not edit the setup cell and do not retrain or replace `ppg_afib_model_augmented.h5`.** When it finishes, *File -> Download -> Download .ipynb* (with the outputs) and send the executed notebook back, together with the CSV that is written to `ppg_afib_project/augmented_window_results/`.
If a number differs from the 'letter' value printed next to it, do **not** change anything to make it match: just send the notebook as it is.

In [3]:
# ============================================================
# SHARED SETUP — AUGMENTED data (4x Cheng et al. 2020), identical to the paper pipeline
# ============================================================
import os, sys, random, glob, json, time, zipfile, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import scipy.signal, scipy.stats
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
def set_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)
set_seeds()

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

PROJECT_DIR = os.environ.get('PPG_PROJECT_DIR', '/content/drive/MyDrive/ppg_afib_project')
OUT_DIR = f'{PROJECT_DIR}/augmented_window_results'
os.makedirs(OUT_DIR, exist_ok=True)

FS = 125
WINDOW_SEC = 20
WINDOW_SAMPLES = FS * WINDOW_SEC   # 2,500 samples

# ---------- 1. FIND + unzip the two PERform CSV zips (searches Drive recursively, like your earlier notebook) ----------
import re
DATA = f'{PROJECT_DIR}/ppg_afib_project'      # folder where you uploaded the two zips (searched first)
AF_ZIP_MANUAL, NON_ZIP_MANUAL = None, None   # OPTIONAL: hard-code the two paths if auto-search picks the wrong files

def _find_zips(root):
    hits = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ('.Trash', '.Trash-0')]
        hits += [os.path.join(dp, f) for f in fns if f.lower().endswith('.zip')]
    return [z for z in hits if 'hls4ml' not in os.path.basename(z).lower()]
_is_non = lambda p: 'non' in os.path.basename(p).lower()
_is_af = lambda p: bool(re.search(r'(^|[^a-z])af([^a-z]|$)', os.path.basename(p).lower())) and not _is_non(p)

if not (os.path.isdir('af_data') and os.path.isdir('non_af_data')):
    if AF_ZIP_MANUAL and NON_ZIP_MANUAL:
        af_zip, non_zip = AF_ZIP_MANUAL, NON_ZIP_MANUAL
    else:
        zips = _find_zips(DATA) or _find_zips(os.path.dirname(PROJECT_DIR))
        print('candidate zips:', *zips, sep='\n  ')
        af_zip = next((z for z in zips if _is_af(z)), None); non_zip = next((z for z in zips if _is_non(z)), None)
    assert af_zip and non_zip, f'AF / non-AF zips not found - set AF_ZIP_MANUAL / NON_ZIP_MANUAL (looked under {DATA})'
    print('AF zip    :', af_zip); print('non-AF zip:', non_zip)
    for zp, d in ((af_zip, 'af_data'), (non_zip, 'non_af_data')):
        with zipfile.ZipFile(zp) as z:
            z.extractall(d)

af_files = sorted(glob.glob('af_data/**/mimic_perform_af_*_data.csv', recursive=True))
non_af_files = sorted(glob.glob('non_af_data/**/mimic_perform_non_af_*_data.csv', recursive=True))
print(f'Found {len(af_files)} AF files | {len(non_af_files)} non-AF files')
assert len(af_files) and len(non_af_files), 'CSV files not found after extraction - check the zip contents'

# ---------- 2. Original (un-augmented) 20 s windows, cached to Drive ----------
def load_windows(file_list, label):
    W, L, S = [], [], []
    for idx, fp in enumerate(file_list):
        ppg = pd.read_csv(fp)['PPG'].values.astype(np.float32)
        ppg = ppg[~np.isnan(ppg)]
        for i in range(len(ppg) // WINDOW_SAMPLES):
            W.append(ppg[i * WINDOW_SAMPLES:(i + 1) * WINDOW_SAMPLES])
            L.append(label); S.append(f'{label}_{idx}')   # same subject-ID scheme as the original notebook
    return W, L, S

CACHE = f'{PROJECT_DIR}/original_windows_cache.npz'
if os.path.exists(CACHE):
    _c = np.load(CACHE, allow_pickle=False)
    W_ALL, L_ALL, S_ALL = _c['W'], _c['L'], _c['S']
else:
    w1, l1, s1 = load_windows(af_files, 1)
    w0, l0, s0 = load_windows(non_af_files, 0)
    W_ALL = np.array(w1 + w0, dtype=np.float32)
    L_ALL = np.array(l1 + l0, dtype=np.int32)
    S_ALL = np.array(s1 + s0)
    np.savez_compressed(CACHE, W=W_ALL, L=L_ALL, S=S_ALL)
print(f'Original windows: {W_ALL.shape} | AF: {(L_ALL==1).sum()} | non-AF: {(L_ALL==0).sum()} | subjects: {len(np.unique(S_ALL))}')

# ---------- 3. Peak detection + the 11 HRV features (identical to original; parameters exposed) ----------
def extract_intervals(ppg, fs=FS, prom_frac=0.15, min_dist_s=0.4, plaus=(0.33, 1.5), plaus_frac=0.8):
    min_distance = int(min_dist_s * fs)
    rng = np.ptp(ppg)
    prom = prom_frac * rng if rng > 0 else 0
    peaks, _ = scipy.signal.find_peaks(ppg, distance=min_distance, prominence=prom)
    if len(peaks) < 2:
        return None
    iv = np.diff(peaks / fs)
    ok = (iv > plaus[0]) & (iv < plaus[1])
    if ok.mean() < plaus_frac:
        return None
    return iv

def compute_features(iv):
    if iv is None or len(iv) < 4:
        return None
    mean_iv, std_iv = float(np.mean(iv)), float(np.std(iv))
    rmssd = float(np.sqrt(np.mean(np.diff(iv) ** 2)))
    mn, mx = float(np.min(iv)), float(np.max(iv))
    cv = std_iv / mean_iv if mean_iv > 0 else 0.0
    d = np.abs(np.diff(iv))
    return [mean_iv, std_iv, rmssd, mn, mx, mx - mn, cv, float(len(iv) + 1),
            float(np.median(iv)), float(scipy.stats.skew(iv)), float(np.mean(d > 0.05))]

FEATURE_NAMES = ['mean_iv', 'std_iv', 'rmssd', 'min_iv', 'max_iv', 'range_iv', 'cv_iv', 'n_beats', 'median_iv', 'skew_iv', 'pnn50']

def build_features(W, L, S, **peak_kw):
    """Returns X, y, subj, kept_index (windows rejected by the quality gate are dropped, as in the original)."""
    X, y, s, kept = [], [], [], []
    for i in range(len(W)):
        f = compute_features(extract_intervals(W[i], **peak_kw))
        if f is not None:
            X.append(f); y.append(L[i]); s.append(S[i]); kept.append(i)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), np.array(s), np.array(kept)

# ---------- 3b. Cheng et al. (2020) augmentation + quality gate, IDENTICAL to data_augmentation_pipeline.ipynb ----------
def apply_cheng_data_augmentation(segment, fs=FS):
    """original + Gaussian noise (5% of window SD) + amplitude scaling N(1, 0.15) + 0.2 Hz baseline wander (20% of window SD)"""
    out = [segment]
    out.append(segment + np.random.normal(0, 0.05 * np.std(segment), len(segment)))
    out.append(segment * np.random.normal(1.0, 0.15))
    t = np.linspace(0, len(segment) / fs, len(segment))
    out.append(segment + 0.2 * np.std(segment) * np.sin(2 * np.pi * 0.2 * t))
    return out

def build_features_augmented(W, L, S):
    X, y, s, dropped = [], [], [], {0: 0, 1: 0}
    for i in range(len(W)):                       # AF windows first, then non-AF (same order as the original notebook)
        for v in apply_cheng_data_augmentation(W[i], fs=FS):
            f = compute_features(extract_intervals(v))
            if f is not None:
                X.append(f); y.append(L[i]); s.append(S[i])
            else:
                dropped[int(L[i])] += 1
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), np.array(s), dropped

np.random.seed(42); random.seed(42)               # same RNG state as CELL 1 of data_augmentation_pipeline.ipynb
X_ALL, y_ALL, subj_ALL, DROPPED = build_features_augmented(W_ALL, L_ALL, S_ALL)
print(f'AUGMENTED feature matrix: {X_ALL.shape} | AF: {(y_ALL==1).sum()} | non-AF: {(y_ALL==0).sum()} | dropped: {DROPPED}')
print('EXPECTED (paper): (7706, 11) | AF 4282 | non-AF 3424 | dropped AF 266, non-AF 404')
if X_ALL.shape[0] != 7706:
    print('WARNING: the dataset differs from the one in the paper. Check the CSV order / that the same two zips are used.')

# ---------- 4. Splits ----------
def holdout_subjects():
    """Same subject-level 80/20 stratified split (random_state=42) as CELL 6 of the original notebook."""
    uniq = np.unique(subj_ALL)
    labs = [y_ALL[subj_ALL == s][0] for s in uniq]
    tr, te = train_test_split(uniq, test_size=0.20, random_state=42, stratify=labs)
    return tr, te

def make_subject_folds(subj_array, label_array, n_folds=5, seed=SEED):
    """Stratified (by class) subject -> fold map. Fixed ONCE so every configuration uses identical test subjects."""
    rng = np.random.RandomState(seed)
    fmap = {}
    for cls in (0, 1):
        subs = np.unique(subj_array[label_array == cls]); rng.shuffle(subs)
        for k, s in enumerate(subs):
            fmap[s] = k % n_folds
    return fmap

FOLD_MAP = make_subject_folds(S_ALL, L_ALL, 5)

# ---------- 5. Model + evaluation helpers ----------
def make_mlp(n_in=11):
    return models.Sequential([layers.Input(shape=(n_in,)),
                              layers.Dense(16, activation='relu'), layers.Dropout(0.2),
                              layers.Dense(8, activation='relu'),
                              layers.Dense(1, activation='sigmoid')])

def fit_mlp(Xtr, ytr, epochs=30, seed=SEED):
    tf.keras.backend.clear_session(); set_seeds(seed)
    m = make_mlp(Xtr.shape[1])
    m.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(Xtr, ytr, epochs=epochs, batch_size=32, verbose=0)
    return m

def keras_fp(Xtr, ytr, Xte, epochs=30):
    sc = StandardScaler(); a = sc.fit_transform(Xtr); b = sc.transform(Xte)
    return fit_mlp(a, ytr, epochs).predict(b, verbose=0).flatten()

def sklearn_fp(factory):
    def fp(Xtr, ytr, Xte):
        sc = StandardScaler(); a = sc.fit_transform(Xtr); b = sc.transform(Xte)
        clf = factory(); clf.fit(a, ytr)
        if hasattr(clf, 'predict_proba'):
            return clf.predict_proba(b)[:, 1]
        return 1.0 / (1.0 + np.exp(-clf.decision_function(b)))
    return fp

def loso_predict(X, y, subj, fit_predict, verbose=True):
    """Leave-one-subject-out: returns out-of-subject probabilities aligned with X."""
    p = np.zeros(len(y), dtype=np.float64)
    uniq = np.unique(subj)
    for k, s in enumerate(uniq):
        te = subj == s
        p[te] = fit_predict(X[~te], y[~te], X[te])
        if verbose and (k + 1) % 5 == 0:
            print(f'  LOSO {k+1}/{len(uniq)}', end='\r')
    return p

def fold_predict(X, y, subj, fit_predict, fmap=None):
    """Fixed subject-level 5-fold CV; returns pooled out-of-fold probabilities + fold id per sample."""
    fmap = fmap or FOLD_MAP
    fold = np.array([fmap[s] for s in subj]); p = np.zeros(len(y))
    for k in range(max(fmap.values()) + 1):
        te = fold == k
        p[te] = fit_predict(X[~te], y[~te], X[te])
    return p, fold

# ---------- 6. Subject-clustered statistics (windows from one subject are NOT independent) ----------
def subject_counts(y, p, subj, thr=0.5):
    pred = (p >= thr).astype(int); rows = []
    for s in np.unique(subj):
        m = subj == s; yy, pp = y[m], pred[m]
        assert len(np.unique(yy)) == 1, 'a subject has mixed labels'
        rows.append((s, int(yy[0]), int(((pp == 1) & (yy == 1)).sum()), int(((pp == 1) & (yy == 0)).sum()),
                     int(((pp == 0) & (yy == 0)).sum()), int(((pp == 0) & (yy == 1)).sum())))
    return pd.DataFrame(rows, columns=['subject', 'label', 'tp', 'fp', 'tn', 'fn'])

def _m(tp, fp, tn, fn):
    n = tp + fp + tn + fn
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    prec = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
    return {'accuracy': (tp + tn) / n, 'sensitivity': sens, 'specificity': spec, 'precision': prec, 'f1': f1}

def cluster_bootstrap(y, p, subj, n_boot=5000, seed=SEED, thr=0.5, do_auc=True, n_boot_auc=1000):
    """Class-stratified subject bootstrap (resample AF subjects among AF, non-AF among non-AF).
    Returns {metric: (point, lo95, hi95)}; also 'per_subject_acc_mean' (macro average over subjects)."""
    rng = np.random.RandomState(seed)
    C = subject_counts(y, p, subj, thr)
    cnt = C[['tp', 'fp', 'tn', 'fn']].values.astype(float)
    af = np.where(C['label'].values == 1)[0]; nf = np.where(C['label'].values == 0)[0]
    point = _m(*cnt.sum(0))
    persub = ((C.tp + C.tn) / (C.tp + C.fp + C.tn + C.fn)).values
    point['per_subject_acc_mean'] = persub.mean()
    draws = {k: [] for k in point}
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(af, len(af)), rng.choice(nf, len(nf))])
        r = _m(*cnt[idx].sum(0)); r['per_subject_acc_mean'] = persub[idx].mean()
        for k in r: draws[k].append(r[k])
    out = {k: (point[k], *np.nanpercentile(draws[k], [2.5, 97.5])) for k in point}
    if do_auc:
        subs = C['subject'].values
        by = {s: np.where(subj == s)[0] for s in subs}
        out['auc'] = (roc_auc_score(y, p), np.nan, np.nan)
        a = []
        for _ in range(n_boot_auc):
            idx = np.concatenate([rng.choice(af, len(af)), rng.choice(nf, len(nf))])
            ii = np.concatenate([by[subs[j]] for j in idx])
            if len(np.unique(y[ii])) == 2:
                a.append(roc_auc_score(y[ii], p[ii]))
        out['auc'] = (out['auc'][0], *np.percentile(a, [2.5, 97.5]))
    return out

def paired_bootstrap_diff(y, pA, pB, subj, n_boot=5000, seed=SEED, thr=0.5):
    """Accuracy(A) - Accuracy(B) with a shared subject resample. Returns (diff, lo95, hi95)."""
    rng = np.random.RandomState(seed)
    CA, CB = subject_counts(y, pA, subj, thr), subject_counts(y, pB, subj, thr)
    a = CA[['tp', 'fp', 'tn', 'fn']].values.astype(float); b = CB[['tp', 'fp', 'tn', 'fn']].values.astype(float)
    af = np.where(CA['label'].values == 1)[0]; nf = np.where(CA['label'].values == 0)[0]
    acc = lambda c: (c[0] + c[2]) / c.sum()
    d = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(af, len(af)), rng.choice(nf, len(nf))])
        d.append(acc(a[idx].sum(0)) - acc(b[idx].sum(0)))
    return acc(a.sum(0)) - acc(b.sum(0)), *np.percentile(d, [2.5, 97.5])

def wilson_ci(k, n, z=1.96):
    """NAIVE window-level Wilson interval — anti-conservative here (windows are clustered by subject)."""
    ph = k / n; den = 1 + z ** 2 / n
    c = (ph + z ** 2 / (2 * n)) / den
    h = z * np.sqrt(ph * (1 - ph) / n + z ** 2 / (4 * n ** 2)) / den
    return c - h, c + h

def fmt(t, pct=True):
    f = 100 if pct else 1
    return f'{t[0]*f:.2f} [{t[1]*f:.2f}, {t[2]*f:.2f}]'

print('Setup complete.')

Mounted at /content/drive
candidate zips:
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_sriramchandar reddy (1).zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment2 sriramchandarreddy.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/S20250020287_ASSIGNMENT2 SRIRAM.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment-3 sriram.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287 cp lab assignment-4.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment-5.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment-6.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_sriramchandar reddy.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_a

In [4]:
from scipy.signal import butter, sosfilt, sosfiltfilt
def bandpass(W, lo, hi, causal):
    sos = butter(4, [lo, hi], btype='band', fs=FS, output='sos')
    return (sosfilt(sos, W.astype(np.float64), axis=1) if causal else sosfiltfilt(sos, W.astype(np.float64), axis=1)).astype(np.float32)

GRID = [  # (name, band-pass (lo, hi, causal) or None, kwargs of extract_intervals)
 ('published *',                         None,           {}),
 ('gate 0.6 *',                          None,           dict(plaus_frac=0.6)),
 ('gate 0.7',                            None,           dict(plaus_frac=0.7)),
 ('gate 0.9',                            None,           dict(plaus_frac=0.9)),
 ('gate 1.0 *',                          None,           dict(plaus_frac=1.0)),
 ('no gate (accept every window)',       None,           dict(plaus_frac=0.0)),
 ('prominence 5% *',                     None,           dict(prom_frac=0.05)),
 ('prominence 10%',                      None,           dict(prom_frac=0.10)),
 ('prominence 25%',                      None,           dict(prom_frac=0.25)),
 ('prominence 35% *',                    None,           dict(prom_frac=0.35)),
 ('min. peak distance 0.33 s',           None,           dict(min_dist_s=0.33)),
 ('min. peak distance 0.5 s',            None,           dict(min_dist_s=0.5)),
 ('plausible interval 0.25-2.0 s',       None,           dict(plaus=(0.25, 2.0))),
 ('plausible interval 0.4-1.2 s',        None,           dict(plaus=(0.4, 1.2))),
 ('band-pass 0.5-8 Hz zero-phase *',     (0.5, 8, False), {}),
 ('band-pass 0.5-8 Hz causal *',         (0.5, 8, True),  {}),
 ('band-pass 0.5-5 Hz zero-phase',       (0.5, 5, False), {}),
 ('band-pass 0.5-5 Hz causal',           (0.5, 5, True),  {}),
 ('gate 1.0 + prominence 25%',           None,           dict(plaus_frac=1.0, prom_frac=0.25)),
 ('band-pass 0.5-8 causal + gate 1.0',   (0.5, 8, True),  dict(plaus_frac=1.0)),
]
assert len(GRID) == 20
n_af, n_non = int((L_ALL == 1).sum()), int((L_ALL == 0).sum())

def cv_accuracy(X, y, subj):
    fold = np.array([FOLD_MAP[s] for s in subj]); accs = []
    for k in range(5):
        te = fold == k
        if te.sum() == 0 or (~te).sum() == 0: continue
        sc = StandardScaler().fit(X[~te]); a = sc.transform(X[~te]).astype(np.float32); b = sc.transform(X[te]).astype(np.float32)
        accs.append(np.mean((fit_mlp(a, y[~te]).predict(b, verbose=0).flatten() >= .5) == y[te]))
    return np.array(accs)

rows = []
for name, bp, kw in GRID:
    Wf = W_ALL if bp is None else bandpass(W_ALL, *bp[:2], causal=bp[2])
    X, y, s, kept = build_features(Wf, L_ALL, S_ALL, **kw)
    accs = cv_accuracy(X, y, s)
    rows.append(dict(setting=name, windows_kept=len(y), pct_AF_kept=100 * (y == 1).sum() / n_af, pct_nonAF_kept=100 * (y == 0).sum() / n_non,
                     cv_accuracy=100 * accs.mean(), fold_sd=100 * accs.std(), n_folds=len(accs)))
    print(rows[-1])
TF_ = pd.DataFrame(rows).round(2); display(TF_)
print('accuracy range over the 20 settings: %.2f - %.2f  (letter 89.02 - 98.23)' % (TF_.cv_accuracy.min(), TF_.cv_accuracy.max()))
TF_.to_csv(f'{OUT_DIR}/filter_sensitivity.csv', index=False)

{'setting': 'published *', 'windows_kept': 1920, 'pct_AF_kept': np.float64(93.75549692172383), 'pct_nonAF_kept': np.float64(89.23719958202717), 'cv_accuracy': np.float64(96.59660709011773), 'fold_sd': np.float64(1.9643414985882672), 'n_folds': 5}
{'setting': 'gate 0.6 *', 'windows_kept': 2017, 'pct_AF_kept': np.float64(97.44942832014073), 'pct_nonAF_kept': np.float64(94.98432601880877), 'cv_accuracy': np.float64(92.98545723763047), 'fold_sd': np.float64(5.007688642466221), 'n_folds': 5}
{'setting': 'gate 0.7', 'windows_kept': 1972, 'pct_AF_kept': np.float64(95.86631486367634), 'pct_nonAF_kept': np.float64(92.16300940438872), 'cv_accuracy': np.float64(94.2845964927281), 'fold_sd': np.float64(3.951426933509785), 'n_folds': 5}
{'setting': 'gate 0.9', 'windows_kept': 1851, 'pct_AF_kept': np.float64(89.79771328056289), 'pct_nonAF_kept': np.float64(86.72936259143155), 'cv_accuracy': np.float64(97.84082651524788), 'fold_sd': np.float64(1.0876294018528871), 'n_folds': 5}
{'setting': 'gate 1.0 

{'setting': 'no gate (accept every window)', 'windows_kept': 2088, 'pct_AF_kept': np.float64(100.0), 'pct_nonAF_kept': np.float64(99.3730407523511), 'cv_accuracy': np.float64(90.08154188167374), 'fold_sd': np.float64(7.98012284602752), 'n_folds': 5}
{'setting': 'prominence 5% *', 'windows_kept': 2047, 'pct_AF_kept': np.float64(99.38434476693052), 'pct_nonAF_kept': np.float64(95.82027168234065), 'cv_accuracy': np.float64(93.62292609351434), 'fold_sd': np.float64(4.8796936739261465), 'n_folds': 5}
{'setting': 'prominence 10%', 'windows_kept': 1999, 'pct_AF_kept': np.float64(97.44942832014073), 'pct_nonAF_kept': np.float64(93.10344827586206), 'cv_accuracy': np.float64(93.6966141844071), 'fold_sd': np.float64(4.632187155338531), 'n_folds': 5}
{'setting': 'prominence 25%', 'windows_kept': 1821, 'pct_AF_kept': np.float64(87.33509234828496), 'pct_nonAF_kept': np.float64(86.52037617554859), 'cv_accuracy': np.float64(97.79347020048654), 'fold_sd': np.float64(0.906597799140444), 'n_folds': 5}


{'setting': 'prominence 35% *', 'windows_kept': 1697, 'pct_AF_kept': np.float64(78.80386983289358), 'pct_nonAF_kept': np.float64(83.69905956112852), 'cv_accuracy': np.float64(97.78318421545184), 'fold_sd': np.float64(0.493828220478701), 'n_folds': 5}
{'setting': 'min. peak distance 0.33 s', 'windows_kept': 1920, 'pct_AF_kept': np.float64(93.75549692172383), 'pct_nonAF_kept': np.float64(89.23719958202717), 'cv_accuracy': np.float64(96.63577918187531), 'fold_sd': np.float64(1.4280011452412815), 'n_folds': 5}
{'setting': 'min. peak distance 0.5 s', 'windows_kept': 1907, 'pct_AF_kept': np.float64(92.78803869832893), 'pct_nonAF_kept': np.float64(89.0282131661442), 'cv_accuracy': np.float64(93.91860594989619), 'fold_sd': np.float64(5.45634808332285), 'n_folds': 5}
{'setting': 'plausible interval 0.25-2.0 s', 'windows_kept': 1982, 'pct_AF_kept': np.float64(96.04221635883906), 'pct_nonAF_kept': np.float64(92.99895506792059), 'cv_accuracy': np.float64(94.34233524796053), 'fold_sd': np.float64(3

,setting,windows_kept,pct_AF_kept,pct_nonAF_kept,cv_accuracy,fold_sd,n_folds
0,published *,1920,93.76,89.24,96.60,1.96,5
1,gate 0.6 *,2017,97.45,94.98,92.99,5.01,5
2,gate 0.7,1972,95.87,92.16,94.28,3.95,5
3,gate 0.9,1851,89.80,86.73,97.84,1.09,5
4,gate 1.0 *,1683,79.07,81.92,98.23,0.97,5
5,no gate (accept every window),2088,100.00,99.37,90.08,7.98,5
6,prominence 5% *,2047,99.38,95.82,93.62,4.88,5
7,prominence 10%,1999,97.45,93.10,93.70,4.63,5
8,prominence 25%,1821,87.34,86.52,97.79,0.91,5
9,prominence 35% *,1697,78.80,83.70,97.78,0.49,5


accuracy range over the 20 settings: 90.08 - 98.58  (letter 89.02 - 98.23)
